Install AJA-pose as a python package building from source

In [1]:
# AJA-pose repository
!git clone https://github.com/kaburia/AJA-pose.git
!cd AJA-pose/code && pip install -e .

Cloning into 'AJA-pose'...
remote: Enumerating objects: 767, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 767 (delta 83), reused 80 (delta 46), pack-reused 625 (from 1)
Receiving objects: 100% (767/767), 143.14 MiB | 11.05 MiB/s, done.
Resolving deltas: 100% (323/323), done.
Updating files: 100% (201/201), done.
Obtaining file:///content/AJA-pose/code
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.1/159.1 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.


Download our model file from Google Cloud Storage


In [2]:
!mkdir /content/AJA-pose/code/aja_pose/modelDir

import urllib.request

# The model file
url = "https://storage.googleapis.com/figures-gp/animal-kingdom/aja_pose_model.pth"
destination = "/content/AJA-pose/code/aja_pose/modelDir/all_animals_no_pretrain_106.pth"

urllib.request.urlretrieve(url, destination)

mkdir: cannot create directory ‘/content/AJA-pose/code/aja_pose/modelDir’: File exists


('/content/AJA-pose/code/aja_pose/modelDir/all_animals_no_pretrain_106.pth',
 <http.client.HTTPMessage at 0x79490e29ecb0>)

In [ ]:
import urllib.request

# The model file
url = "https://storage.googleapis.com/figures-gp/animal-re-id/data/WhatsApp%20Video%202024-06-12%20at%2009.24.48.mp4"
destination = "/content/AJA-pose/code/aja_pose/dataDir/llama2.mp4"

urllib.request.urlretrieve(url, destination)

('/content/AJA-pose/code/aja_pose/dataDir/llama2.mp4',
 <http.client.HTTPMessage at 0x7fcd4170d7e0>)

In [3]:
!pip install ultralytics

## Animals Grazing Tests

In [1]:
import os
# os.mkdir('AnimalsGrazing')
dir = os.getcwd()

# !gsutil -m cp -r gs://figures-gp/AnimalsGrazing {dir}
!gsutil -m cp -r gs://figures-gp/MegaDetector {dir}

Copying gs://figures-gp/MegaDetector/MDV6b-yolov9c.pt...


In [ ]:
# # Polar bears video
# url = "https://storage.googleapis.com/figures-gp/animal-re-id/data/PolarBearVidID.zip"
# destination = "/content/PolarBearVidID.zip"

# urllib.request.urlretrieve(url, destination)

('/content/PolarBearVidID.zip', <http.client.HTTPMessage at 0x7e272f41d4e0>)

In [ ]:
# !unzip /content/PolarBearVidID.zip -d /content

In [2]:
!pip install --upgrade ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.7/898.7 kB 23.9 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.0.196
    Uninstalling ultralytics-8.0.196:
      Successfully uninstalled ultralytics-8.0.196
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aja-pose 0.0.1 requires ultralytics==8.0.196, but you have ultralytics 8.3.49 which is incompatible.


In [3]:
import os
os.chdir('/content/AJA-pose/code')

In [4]:
!ls

aja_pose				   object-detection  run_test.ipynb  tests.py
aja_pose.egg-info			   output	     setup.py	     yolov8n.pt
build					   polarvid.ipynb    steps.md	     yolov8s.pt
MANIFEST.in				   README.md	     steps.txt
my_module.cpython-310-x86_64-linux-gnu.so  requirements.txt  tests.ipynb


Restart session before running next cell

In [8]:
from aja_pose import Model

import os
import yaml
import sys
import warnings
import os.path as osp

import torch
import torchvision.transforms as transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO
from tqdm import tqdm
import multiprocessing as mp
import json
import logging
from pathlib import Path
import gc
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn.functional as F




warnings.filterwarnings("ignore")
# get the current path
module_dir = os.getcwd()

# polar bears path
polar_bears_path = os.path.join(Path(module_dir).parent.parent.parent.parent,
                                    'gait-tests', 'run-gsp', 'data', 'PolarBearVid', '000')
landmarks_000 = os.path.join(Path(polar_bears_path).parent.parent, 'PolarBearLandmarks', '000')

# Initialize logging
logging.basicConfig(filename='/content/000.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')


 # yaml file path
yaml_file = os.path.join(module_dir, 'aja_pose', 'experiments', 'mpii', 'vhrbirdpose', 'w32_256x256_adam_lr1e-3_ak_vhr_s.yaml')
pretrained_model = os.path.join(module_dir, 'aja_pose', 'modelDir', 'aja_pose_model.pth')
yolo_model = os.path.join(module_dir, 'aja_pose', 'modelDir', 'yolo_det_best.pt')

def add_path(path):
    if path not in sys.path:
        sys.path.insert(0, path)

# this_dir = osp.dirname(__file__)

lib_path = osp.join(module_dir, 'aja_pose', 'lib')
add_path(lib_path)

import models

from aja_pose.lib.config import cfg
from aja_pose.lib.core.inference import *
from aja_pose.lib.utils.transforms import transform_preds
# Import the Sort Algorithm
# from aja_pose.sort import Sort
import concurrent.futures

class Args:
    def __init__(self, cfg, opts=[],modelDir='', logDir='', dataDir='', prevModelDir=''):
        self.cfg = cfg
        self.opts = opts
        self.modelDir = modelDir
        self.logDir = logDir
        self.dataDir = dataDir
        self.prevModelDir = prevModelDir

class ImageDataset(Dataset):
    def __init__(self, images_directory, transform=None):
        self.images_directory = images_directory
        self.image_files = os.listdir(images_directory)
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = os.path.join(self.images_directory, self.image_files[idx])
        # print(img_name)
        try:
            image = Image.open(img_name).convert('RGB')
        except Exception as e:
            logging.error(f"Cannot identify image file {img_name} with Exception {e}")
            # Return a dummy image instead of None to avoid the error
            return torch.zeros((3, 256, 256)), self.image_files[idx]

        if self.transform:
            image = self.transform(image)

        return image, self.image_files[idx]


# path to data directory
class PoseInference(Model): # Inherit from the Model class
    # add path to lib directory
    def add_path(self):
        lib_path = osp.join(module_dir, 'lib')
        if lib_path not in sys.path:
            sys.path.insert(0, lib_path)

    def load_model(self, pretrained=pretrained_model):
        sys.path.append(os.path.join(module_dir, 'lib'))
        from aja_pose.lib.config import cfg
        from aja_pose.lib.config import update_config


        super().write_yaml(yaml_file, pretrained=pretrained)

        if torch.cuda.is_available():
            device = torch.device('cuda')
        else:
            device = torch.device('cpu')

        args = Args(yaml_file)

        update_config(cfg, args)

        # load the model
        model = eval('models.'+cfg.MODEL.NAME+'.get_pose_net')(
            cfg, is_train=False
        )

        # load the model weights
        model.load_state_dict(torch.load(cfg.TEST.MODEL_FILE, map_location=device), strict=False)
        model.eval()

        return model
    # load image
    def load_image(self, image_path, transform=True):
        # check if image path is a string or an image
        if isinstance(image_path, str):
            # load the image
            image = Image.open(image_path).convert('RGB')
        elif isinstance(image_path, np.ndarray):
            image = Image.fromarray(image_path)
        elif isinstance(image_path, Image.Image):
            image = image_path
        else:
            raise ValueError('Image path must be a string or an image')
         # image transformation
        image_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        if transform:
            image = image_transform(image)
            # add batch dimension
            image = image.unsqueeze(0)
        return image

    # Image inference
    def image_pose(self, image_path, model=pretrained_model, thresh=0.5,):
        # load the model
        model = self.load_model(pretrained=model)
        # load the transformed image
        image_t = self.load_image(image_path)
        # load the original image
        image = self.load_image(image_path, transform=False)
        image_size = image.size
        # get the keypoints from the model
        with torch.no_grad():
            output = model(image_t) # shape (1,23,64,64) Heatmap
                # get the image centre
        def get_image_center():
            return 128,128

        # get the image scale with respect to 200px
        def get_image_scale():
            from math import sqrt
            width = 256
            height = 256
            return 200/256

        center = list(get_image_center())
        scale = get_image_scale()

        cent = np.array([[center]])
        sc = [scale]
        # print(sc)
        # coords, maxvals = get_final_preds(
        #                 cfg, output.clone().cpu().numpy(), cent, sc)
        coords, maxvals = get_max_preds(output.clone().cpu().numpy())
        # print(coords)
        # convert image to numpy
        # img = image.numpy().transpose(1,2,0)
        # resize to the original image size
        # img = cv2.resize(img, image_size)
        # thresh = 0.5
        # remove normalization
        # img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        # img = np.clip(img, 0, 1)
        # subset maxvals > thresh
        # maxvals = np.array([np.array([i for i in maxvals[0] if i >= thresh])])

        return coords, maxvals

    def batch_processing(self, images, thresh=0.5, model=pretrained_model):
        """
        Performs pose estimation on a batch of images.

        Args:
            images (torch.Tensor): A batch of images.
            thresh (float, optional): Confidence threshold for keypoints. Defaults to 0.5.
            model (torch.nn.Module, optional): Pose estimation model. Defaults to self.model.

        Returns:
            tuple: A tuple of two lists:
                - all_coords: List of keypoint coordinates for each image in the batch.
                - all_maxvals: List of keypoint confidence scores for each image in the batch.
        """

        # if model is None:
        #     model = self.model
        # load the model
        model = self.load_model(pretrained=model)

        if torch.cuda.is_available():
            device = torch.device('cuda')
        else:
            device = torch.device('cpu')

        model.to(device)

        with torch.no_grad():
            output = model(images)
            coords, maxvals = get_max_preds(output.clone().cpu().numpy())

        return coords, maxvals

    def draw_pose(self, image_path, thresh=0.1, model=pretrained_model):
        # load the image
        image = self.load_image(image_path, transform=False)
        # get the coords and maxvals
        coords, maxvals = self.image_pose(image_path, thresh=thresh, model=model)
        plt.imshow(image)
        plt.title('Image and predicted keypoints')
        plt.axis('off')

        # Plot the keypoints
        for i in range(len(coords[0])):  # Iterate through each keypoint
            x, y = coords[0][i]
            # x,y = x/256*image_size[0], y/256*image_size[1]
            x,y = x*image.size[0]/64, y*image.size[1]/64
            # print(x,y)
            if maxvals[0][i] < thresh:
                continue

            plt.scatter(x,y, color='red', s=10)  # Plot the keypoint on the image
        plt.show()

# object detection
class ObjectDetection:
    def load_model(self, model_path='/content/MegaDetector/MDV6b-yolov9c.pt'):
        # Add the option to load to GPU if available
        if torch.cuda.is_available():
            device = torch.device('cuda')
        else:
            device = torch.device('cpu')
        # Load the model

        model = YOLO(model_path)
        model.to(device)
        # print(model)
        return model
    def image_detection(self, image_path, model='/content/MegaDetector/MDV6b-yolov9c.pt', stream=False):
        model = self.load_model(model_path=model)
        # load image
        if isinstance(image_path, str):
            image = Image.open(image_path).convert('RGB')
        elif isinstance(image_path, np.ndarray):
            image = Image.fromarray(image_path)
        elif isinstance(image_path, Image.Image):
            image = image_path
        # image = Image.open(image_path).convert('RGB')
        results = model.predict(image, stream=stream)
        return results

# handling data
class HandlingData:
    def convert_to_serializable(self, result):
        # result is a dict containing ndarray objects that need conversion
        serializable_result = {}
        for key, value in result.items():
            if isinstance(value, torch.Tensor):
                serializable_result[key] = value.tolist()
            elif isinstance(value, np.ndarray):
                serializable_result[key] = value.tolist()
            elif isinstance(value, dict):
                serializable_result[key] = self.convert_to_serializable(value)
            elif isinstance(value, list):
                serializable_result[key] = [self.convert_to_serializable(v) if isinstance(v, (dict, np.ndarray, torch.Tensor)) else v for v in value]
            else:
                serializable_result[key] = value
        return serializable_result

    def append_to_json_file(self, filepath, new_data):
        try:
            # if os.path.exists(filepath):
            #     with open(filepath, 'r') as f:
            #         existing_data = json.load(f)
            # else:
            #     existing_data = {}
            with open(filepath, 'a') as f:
                for key, value in new_data.items():
                    f.write(json.dumps({key: value}, indent=4) + "\n")
            logging.info(f"Results appended to {filepath}")


            # existing_data.update(new_data)

            # with open(filepath, 'a') as f:
            #     json.dump(existing_data, f, indent=4)
        except Exception as e:
            logging.error(f"Error appending to JSON file {filepath}: {e}")

# combine the two classes
class DetectionPoseInference(ObjectDetection):
    def __init__(self):
        self.det_model = self.load_model()
        # self.image_detect = self.image_detection()
        # initialize pose model
        self.pose_model = PoseInference()
        self.serialize = HandlingData()

    # handle detection by passing in an image and passing out the detected images bounding boxes
    def detect_objects(self, image_path, model='/content/MegaDetector/MDV6b-yolov9c.pt', stream=False):
        results = self.image_detection(image_path, model=model, stream=stream)

        return results[0]

    def fit_pose(self, image_path, thresh=0.1, bbox_thresh=0.1, model=pretrained_model,
                  stream=False, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', video=False):
            """
            Processes an image/frame with object detection and pose estimation using GPU acceleration.

            Args:
                image_path (str or np.ndarray or Image.Image): Path to the image/frame.
                thresh (float, optional): Threshold for pose estimation confidence. Defaults to 0.5.
                bbox_thresh (float, optional): Threshold for bounding box confidence. Defaults to 0.5.
                model (torch.nn.Module): Pre-trained pose estimation model.
                stream (bool, optional): Whether processing a video stream. Defaults to False.
                detection_model (str, optional): Path to the detection model. Defaults to '/content/MegaDetector/MDV6b-yolov9c.pt'.

            Returns:
                dict: Dictionary containing class IDs, bounding boxes, pose coordinates, and pose confidences.
            """

        # try:
            # Get bounding boxes
            result = self.detect_objects(image_path, stream=stream, model=detection_model)

            # Handle empty results
            if not result.boxes:
                return None

            # Convert image to a torch tensor
            if isinstance(image_path, str):
                image = Image.open(image_path).convert('RGB')
                # Keep a copy of the original PIL image for cropping
                original_image = image
                image = torch.from_numpy(np.array(image)).unsqueeze(0)  # Add batch dimension
            elif isinstance(image_path, np.ndarray):
                image = torch.from_numpy(image_path).unsqueeze(0)  # Add batch dimension
                original_image = Image.fromarray(image_path)  # Create PIL image from ndarray
            elif isinstance(image_path, Image.Image):
                image = torch.from_numpy(np.array(image_path)).unsqueeze(0)  # Add batch dimension
                original_image = image_path  # Keep the original PIL image
            else:
                raise ValueError("Unsupported image format")

            # Image processing logic
            if not video:

              # Move image to GPU (if available)
              if torch.cuda.is_available():
                  image = image.cuda()

              # Filter detections based on confidence threshold
              filtered_boxes = [box for box in result.boxes]

              # Extract bounding box coordinates
              if filtered_boxes:
                  bbox_coords = torch.stack([box.xyxy[0] for box in filtered_boxes])

              cropped_images = []
              target_size = (256, 256)  # Adjust the target size as needed
              for i, box in enumerate(filtered_boxes):
                x1, y1, x2, y2 = box.xyxy[0].int()
                # Check if the bounding box has valid dimensions
                if (x2 - x1) > 0 and (y2 - y1) > 0:
                    # Crop using the original PIL image
                    cropped_image = original_image.crop((int(x1), int(y1), int(x2), int(y2)))

                    # Convert the cropped image to a tensor and resize
                    cropped_image = transforms.ToTensor()(cropped_image) # Convert to tensor
                    cropped_image = F.interpolate(cropped_image.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0) # Resize
                    cropped_images.append(cropped_image)
                else:
                    logging.warning(f"Skipping bounding box with invalid dimensions: {box.xyxy[0]}")
              # Convert the list of cropped images to a tensor
              cropped_images = torch.stack(cropped_images, dim=0)
              print(cropped_images.shape)

              # Move cropped images to GPU (if available)
              if torch.cuda.is_available():
                  cropped_images = cropped_images.cuda()

              # print(cropped_images)

              # Perform pose estimation on cropped images (batch processing)
              all_coords, all_maxvals = self.pose_model.batch_processing(cropped_images, thresh=thresh, model=model)

              # Combine results with bounding box information
              frame_data = dict()
              for i, box in enumerate(filtered_boxes):
                  class_id = result.names[box.cls[0].item()]
                  bbox_data = {
                      'class_id': class_id,
                      'coords': box.xyxy[0].tolist(),  # Convert back to list
                      'threshold': box.conf[0].item(),
                      'pose_coords': all_coords[i].tolist(),
                      'pose_maxvals': all_maxvals[i].tolist(),
                  }
                  frame_data['bbox_' + str(i)] = bbox_data

              return frame_data
            elif video:
              # Logic to process the video to maximise on GPU resources



        # except Exception as e:
        #     logging.error(f"Error processing {image_path}: {e}")
        #     return None

        # # Get the deteted bounding boxes and pass them to the pose model
    # def fit_pose(self, image_path, thresh=0.5, bbox_thresh=0.5, model=pretrained_model,
    #              stream=False, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt'):
    #     try:
    #         # append the coords and maxvals to a list
    #         pose_coords = []
    #         pose_maxvals = []
    #         # get the data to a dictionary for each bounding box in frame
    #         frame_data = dict()
    #         # get the bounding boxes
    #         result = self.detect_objects(image_path, stream=stream, model=detection_model)
    #         print(result)
    #         # can be an image or a video frame check if string or image
    #         if isinstance(image_path, str):
    #             image = Image.open(image_path).convert('RGB')
    #         elif isinstance(image_path, np.ndarray):
    #             image = Image.fromarray(image_path)
    #         elif isinstance(image_path, Image.Image):
    #             image = image_path
    #         count = 0
    #         for box in result.boxes:
    #             bbox_data = dict()
    #             class_id = result.names[box.cls[0].item()]
    #             # if class_id != 'bear':
    #             #     logging.info(f'Noisy class: {class_id} for image path {image_path}')
    #             #     continue
    #             cords = box.xyxy[0].tolist()
    #             cords = [round(x) for x in cords]
    #             conf = round(box.conf[0].item(), 2)
    #             # print("Object type:", class_id)
    #             # print("Coordinates:", cords)
    #             # print("Probability:", conf)
    #             # print("---")
    #             if conf < bbox_thresh:
    #                 continue
    #             x1, y1, x2, y2 = cords
    #             cropped_image = image.crop((int(x1), int(y1), int(x2), int(y2)))
    #             coords, maxvals = self.pose_model.image_pose(cropped_image, thresh=thresh, model=model)
    #             bbox_data['class_id'] = class_id
    #             bbox_data['coords'] = cords
    #             bbox_data['threshold'] = conf
    #             bbox_data['pose_coords'] = coords
    #             bbox_data['pose_maxvals'] = maxvals
    #             frame_data['bbox_'+str(count)] = bbox_data
    #             count += 1
    #         return frame_data
    #     except Exception as e:
    #         # print(f"Error processing {image_path}: {e}")
    #         logging.error(f"Error processing {image_path}: {e}")
    #         return None
        # # pass the cropped image to the pose model based on the bounding boxes if any
        # for box in boxes:
        #     x1, y1, x2, y2 = box
        #     # crop the image
        #     # cropped_image = image[int(y1):int(y2), int(x1):int(x2)]
        #     cropped_image = image.crop((int(x1), int(y1), int(x2), int(y2)))
        #     # get the pose coords and maxvals for each cropped image
        #     # self.pose_model.image_pose(cropped_image, thresh=thresh)
        #     coords, maxvals = self.pose_model.image_pose(cropped_image, thresh=thresh, model=model)
        #     pose_coords.append(coords)
        #     pose_maxvals.append(maxvals)
        #     print('Detected keypoints:', coords)
        # return pose_coords, pose_maxvals

    # def draw_detection_pose(self, image_path, thresh=0.1, bbox_thresh=0.5, model=pretrained_model):
    #     # load the image
    #     image = Image.open(image_path).convert('RGB')
    #     # get the frame data for all the bounding boxes
    #     frame_data = self.fit_pose(image_path, thresh=thresh, bbox_thresh=bbox_thresh, model=model)
    #     plt.imshow(image)
    #     plt.title('Image and predicted keypoints')
    #     plt.axis('off')
    #     print('Bounding boxes detected:', len(frame_data))
    #     print('---')
    #     print('Bounding box data:', frame_data)

    #     # get the data for each bounding box in the frame and plot all the keypoints for each bounding box as well
    #     for key in frame_data.keys():
    #         print('Bounding box:', key)
    #         bbox_data = frame_data[key]
    #         class_id = bbox_data['class_id']
    #         cords = bbox_data['coords']
    #         pose_coords = bbox_data['pose_coords']
    #         pose_maxvals = bbox_data['pose_maxvals']
    #         print('Class:', class_id)
    #         print('Bounding box:', cords)
    #         print('Pose keypoints:', pose_coords)
    #         print('Pose maxvals:', pose_maxvals)
    #         print('---')
    #         # get the width and height of the bounding box
    #         width = cords[2] - cords[0]
    #         height = cords[3] - cords[1]
    #         # Plot the keypoints
    #         for i in range(len(pose_coords[0])):  # Iterate through each keypoint
    #             x, y = pose_coords[0][i]
    #             print(x,y)
    #             print('---')
    #             print(pose_maxvals[0][i])
    #             x,y = x*width/64 + cords[0], y*height/64 + cords[1]

    #             if pose_maxvals[0][i] < thresh:
    #                 continue
    #             plt.scatter(x,y, color='red', s=10)  # Plot the keypoint on the image
    #     plt.show()

    # # Perform batch inference on the data (directory with images)
    # saves a json file with labels with the names of the images from directory
    # def batch_processing(self, images_directory, thresh=0.5, bbox_thresh=0.5, model=pretrained_model,
    #                         stream=False, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', output_path=None):
    #     # initialize the dictionary to store the data
    #     output_json = dict()
    #     # check if the image directory path exists
    #     assert os.path.exists(images_directory)
    #     # loop through the images in the images directory
    #     for image in os.listdir(images_directory):
    #         # fit the images to the pose model
    #         output_json[image] = self.fit_pose(
    #             image_path=os.path.join(images_directory, image), thresh=thresh, bbox_thresh=bbox_thresh,
    #             model=model, stream=stream, detection_model=detection_model
    #         )

    #     # # save the output json if not None
    #     # if output_path is not None:
    #     #     with
    #     return output_json

    # # Parallel process depending on the device (cpu or gpu)
    # def parallel_inferencing(self, images_directory, cpu_count=3, thresh=0.5, bbox_thresh=0.5, model=pretrained_model,
    #                         stream=False, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', batch_size=4, output_path=None):

    #     # output_json = dict()
    #     # check if gpu is available
    #     if torch.cuda.is_available():
    #         # process to handle the loop effectively on all cuda cores
    #         logging.info("Using GPU for processing.")
    #         transform = transforms.Compose([
    #             transforms.Resize((256, 256)),
    #             transforms.ToTensor(),
    #             transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    #         ])
    #         dataset = ImageDataset(images_directory, transform=transform)
    #         dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=cpu_count)

    #         output_json = dict()
    #         with tqdm(total=len(dataloader), desc='Processing batches on GPU') as batch_pbar:
    #           for images, image_files in dataloader:
    #               images = images.cuda()
    #               with tqdm(total=len(images), desc='Processing images in batch') as image_pbar:
    #                   for i, image in enumerate(images):
    #                       image_path = os.path.join(images_directory, image_files[i])
    #                       frame_data = self.fit_pose(image_path, thresh, bbox_thresh, model, stream, detection_model)
    #                       output_json[image_files[i]] = self.serialize.convert_to_serializable(frame_data)
    #                       image_pbar.update(1)  # Update the inner progress bar
    #               batch_pbar.update(1)  # Update the outer progress bar

    #         if output_path:
    #             self.serialize.append_to_json_file(output_path, output_json)

    #         return output_json
    #     else: # using cpu
    #         # print("Using CPU for processing.")
    #         logging.info("Using CPU for processing.")
    #         # with concurrent.futures.ProcessPoolExecutor(max_workers=cpu_count) as executor:
    #         #     futures = {
    #         #         executor.submit(self.fit_pose, os.path.join(images_directory, image), thresh, bbox_thresh, model, stream, detection_model): image
    #         #         for image in os.listdir(images_directory)
    #         #     }
    #         #     output_json = {futures[future]: future.result() for future in concurrent.futures.as_completed(futures)}
    #         # return output_json
    #         pool = mp.Pool(processes=cpu_count)

    #         try:
    #             count = 0
    #             results = []
    #             image_files = os.listdir(images_directory)
    #             with tqdm(total=len(image_files), desc='Processing images') as pbar:
    #                 for image in image_files:
    #                     results.append(
    #                         pool.apply_async(
    #                             self.fit_pose,
    #                             args=(os.path.join(images_directory, image), thresh, bbox_thresh, model, stream, detection_model),
    #                             callback=lambda _: pbar.update(1)
    #                         )
    #                     )

    #                     if count % 100 == 0 and output_path or count==len(image_files):
    #                         # print(f'Checkpoint {count}')
    #                         logging.info(f'Checkpoint at image {count}')
    #                         output_json = {image_files[i]: self.serialize.convert_to_serializable(result.get()) for i, result in enumerate(results) if result.get() is not None}
    #                         # with open(output_path, 'a') as f:
    #                         #     json.dump(output_json, f, indent=4)
    #                         self.serialize.append_to_json_file(output_path, output_json)
    #                         # Clear results and run garbage collection
    #                         results.clear()
    #                         gc.collect()

    #                     count += 1

    #                 pool.close()
    #                 pool.join()

            #     # output_json = {image_files[i]: self.serialize.convert_to_serializable(result.get()) for i, result in enumerate(results)}

            #     # if output_path:
            #     #     # with open(output_path, 'w') as f:
            #     #     #     json.dump(output_json, f, indent=4)
            #     #     return output_json

            #     return output_json

            # except Exception as e:
            #     # print(f"An error occurred: {e}")
            #     logging.error(f"An error occurred: {e}")
            # finally:
            #     pool.terminate()




# Class to process videos
class VideoProcessor:
    def __init__(self):
        self.fit_pose = DetectionPoseInference()
        self.serialize = HandlingData()

    # Read video frames or live feed
    def read_video(self, video_path): # video_path is the path to the video or 0/1 for live feed depending on camera
        cap = cv2.VideoCapture(video_path)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                raise ValueError('Error reading video')
            yield frame
        cap.release()

    # get video metadata
    def get_video_metadata(self, video_path):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Release the video capture object
        cap.release()

        return fps, frame_count, frame_width, frame_height

    # draw bounding box in frame
    def draw_bbox(self, frame, results, thresh=0.1):
      # Draw the bounding boxes and keypoints on the frame
      for key in results.keys():
          bbox_data = results[key]
          cords = bbox_data['coords']
          pose_coords = bbox_data['pose_coords']
          pose_maxvals = bbox_data['pose_maxvals']
          # print(pose_maxvals[0])
          # print(len(pose_maxvals))
          # get the width and height of the bounding box
          width = cords[2] - cords[0]
          height = cords[3] - cords[1]
          # Plot the keypoints
          for i in range(len(pose_coords)):
              x, y = pose_coords[i]
              x,y = x*width/64 + cords[0], y*height/64 + cords[1]
              if pose_maxvals[i][0] < thresh:
                  continue
              cv2.circle(frame, (int(x), int(y)), 3, (0, 255, 0), -1)
              cv2.rectangle(frame, (int(cords[0]), int(cords[1])), (int(cords[2]), int(cords[3])), (0, 255, 0), 2)
              cv2.putText(frame, bbox_data['class_id'], (int(cords[0]), int(cords[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)


    # def process_video(self, video_path, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', pose_model=pretrained_model,
    #                   thresh=0.1, bbox_thresh=0.5, output_path='video.avi', output_json=None, batch_size=16):
    #     """
    #     Processes a video with object detection and pose estimation using GPU acceleration.

    #     Args:
    #         video_path (str): Path to the input video.
    #         detection_model (str, optional): Path to the detection model. Defaults to '/content/MegaDetector/MDV6b-yolov9c.pt'.
    #         pose_model (torch.nn.Module): Pre-trained pose estimation model.
    #         thresh (float, optional): Threshold for object detection confidence. Defaults to 0.1.
    #         bbox_thresh (float, optional): Threshold for bounding box confidence. Defaults to 0.5.
    #         output_path (str, optional): Path to the output video. Defaults to 'video.avi'.
    #         output_json (str, optional): Path to the output JSON file (optional). Defaults to None.
    #         batch_size (int, optional): Batch size for processing frames. Defaults to 16.
    #     """
    #     import queue
    #     import threading
    #     print(f'Processing video... {video_path}')


    #     # Get video metadata
    #     fps, frame_count, frame_width, frame_height = self.get_video_metadata(video_path)
    #     frame_size = (frame_width, frame_height)

    #     # Define video writer
    #     fourcc = cv2.VideoWriter_fourcc(*'XVID')
    #     video_writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

    #     # Move models to GPU (if available)
    #     # if torch.cuda.is_available():
    #     #     detection_model.cuda()
    #     #     pose_model.cuda()

    #     # Create a queue to store frames
    #     frame_queue = queue.Queue()

    #     # Define a function to read frames asynchronously
    #     def read_frames():
    #         for frame in self.read_video(video_path):
    #             frame_queue.put(frame)

    #     read_thread = threading.Thread(target=read_frames)
    #     read_thread.start()

    #     output_json_dict = dict()  # Dictionary to store results for JSON output

    #     # Process frames in batches
    #     while not frame_queue.empty():
    #         # Get a batch of frames
    #         batch = []
    #         for _ in range(batch_size):
    #             if frame_queue.empty():
    #                 break  # Handle end of video
    #             batch.append(frame_queue.get())

    #         # Convert batch to torch tensors on GPU (if available)
    #         if torch.cuda.is_available():
    #             batch = torch.stack([torch.from_numpy(frame).cuda() for frame in batch])

    #         # Process the batch on GPU
    #         batch_results = self.fit_pose.batch_processing(batch, thresh, bbox_thresh, pose_model, detection_model)

    #         # Process individual results and write to video
    #         for frame, result in zip(batch, batch_results):
    #             self.draw_bbox(frame, result)  # Draw bounding boxes and keypoints
    #             video_writer.write(frame)

    #             # Prepare data for JSON output (optional)
    #             if output_json:
    #                 output_json_dict[frame] = self.serialize.convert_to_serializable(result)

    #     read_thread.join()
    #     video_writer.release()

    #     # Save JSON output (optional)
    #     if output_json:
    #         with open(output_json, 'w') as f:
    #             json.dump(output_json_dict, f, indent=4)

    #     cv2.destroyAllWindows()
    # Process video frames
    def process_video(self, video_path, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', pose_model=pretrained_model,
                      thresh=0.1, bbox_thresh=0.5, output_path='video.avi', output_json=None, initial_batch_size=16,
                      max_batch_size=32, target_fps=30):

        # initial_batch_size = 16  # Adjust based on GPU memory and desired speed
        # max_batch_size = 32  # Maximum batch size to avoid memory issues
        # min_batch_size = 4  # Minimum batch size to ensure efficient GPU utilization
        # target_fps = 30  # Target FPS for video processing
        # get the video frames
        video = self.read_video(video_path)

        # output json file
        output_json = dict()

        # Define the video codec and create a VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'XVID')

        # read the fps and frame size from the video
        fps, frame_count, frame_width, frame_height = self.get_video_metadata(video_path)
        frame_size = (frame_width, frame_height)  # Frame size (width, height)

        frame_batch = []
        # video_fps, frame_count, _, _ = self.get_video_metadata(video_path)
        frame_interval = 1.0 / fps

        video_writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

        # Iterate over the frames and write them to the video file
        # for frame in video:

        for frame in video:
            # every frame goes through the detection model
            results = self.fit_pose.fit_pose(frame, thresh=thresh, bbox_thresh=bbox_thresh,
                                             model=pose_model, detection_model=detection_model) # dict of bounding boxes and keypoints

            print(results)

            # Append the results
            output_json[frame] = self.serialize.convert_to_serializable(results)

            # Draw the boxes and poses
            self.draw_bbox(frame, results)

            # print(results)

                    # put text on top of the bounding box

            video_writer.write(frame)
            # cv2.imshow('Tracking', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        # save the output to a file
        if output_json:
            output_path_json = output_json
        elif output_json is None:
          output_path_json = 'output.json'

        with open(output_path_json, 'w') as f:
                json.dump(output_json, f, indent=4)


        # Release the video writer and close the video file
        video_writer.release()
        cv2.destroyAllWindows()
import time

def process_video(self, video_path, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', pose_model=pretrained_model,
                 thresh=0.1, bbox_thresh=0.5, output_path='video.avi', output_json=None):
    # ... other code

    initial_batch_size = 16  # Adjust based on GPU memory and desired speed
    max_batch_size = 32  # Maximum batch size to avoid memory issues
    min_batch_size = 4  # Minimum batch size to ensure efficient GPU utilization
    target_fps = 30  # Target FPS for video processing

    frame_batch = []
    video_fps, frame_count, _, _ = self.get_video_metadata(video_path)
    frame_interval = 1.0 / video_fps

    start_time = time.time()
    for frame_idx, frame in enumerate(video):
        # ... detection code
        if results:  # Process frame if detections are present
            frame_batch.append((frame, frame_idx))

            # Dynamic batch size adjustment
            if len(frame_batch) >= initial_batch_size:
                num_detections = sum(len(result.boxes) for _, result in frame_batch)
                if num_detections > 100:  # Adjust threshold as needed
                    batch_size = min(max_batch_size, initial_batch_size // 2)
                else:
                    batch_size = min(max_batch_size, initial_batch_size)

                if len(frame_batch) >= batch_size:
                    # Process the batch
                    batch_frames, batch_indices = zip(*frame_batch)
                    batch_results = self.fit_pose(batch_frames)

                    # Extract results and update output data structure
                    for result, idx in zip(batch_results, batch_indices):
                        output_json[idx] = self.serialize.convert_to_serializable(result)
                        self.draw_bbox(frame, result)
                        video_writer.write(frame)

                    # Clear the batch and adjust batch size based on processing time
                    frame_batch = []
                    elapsed_time = time.time() - start_time
                    processed_frames = frame_idx + 1
                    actual_fps = processed_frames / elapsed_time

                    if actual_fps < target_fps * 0.8:  # Adjust threshold as needed
                        batch_size = max(min_batch_size, batch_size // 2)
                    elif actual_fps > target_fps * 1.2:
                        batch_size = min(max_batch_size, batch_size * 2)

                    start_time = time.time()

        # ... remaining code for processing individual frames if no batch is full

    # ... remaining code for video processing

import time
from pathlib import Path

if __name__ == '__main__':
    start = time.time()
    inf = PoseInference()
    obj_det = ObjectDetection()
    pose_det = DetectionPoseInference()
    # obj_track = ObjectTracking()
    vid_process = VideoProcessor()


    # data directory
    dir_data_path = os.path.join(module_dir, 'aja_pose', 'dataDir')
    # image path
    img_path = os.path.join(dir_data_path, 'licensed-image.jfif')
    # video path
    vid_path = os.path.join(dir_data_path, 'sheep_grazing.mp4')

    # Handle video input
    # video folder path
    video_folder = '/content/AnimalsGrazing'
    video_files = [os.path.join(video_folder, f) for f in os.listdir(video_folder) if f.endswith('.mp4') and f != '20240625_134814.mp4']
    # print(video_files[0])
    # inference on the first input
    vid_process.process_video(video_path='/content/AnimalsGrazing/grazing-animals - Made with Clipchamp.mp4')


    # polar bears path
    # polar_bears_path = os.path.join(Path(dir_data_path).parent.parent.parent.parent.parent.parent,
    #                                 'gait-tests', 'run-gsp', 'data', 'PolarBearVid', '000')
    # output_polar_bears_path = os.path.join(Path(polar_bears_path).parent.parent, 'PolarBearLandmarks', '000', '000.json')

    # colab polar bear
    # colab_polar_bears_path = '/content/000'

    # output_json = pose_det.parallel_inferencing(colab_polar_bears_path, cpu_count=4, detection_model='yolov8l.pt',
    #                                             output_path='000.json', batch_size=500)
    # print(output_json)

    # print(output_json)
    end = time.time()
    print(f'Time it took {end-start}')


0: 384x640 15 animals, 37.4ms
Speed: 2.6ms preprocess, 37.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
torch.Size([15, 3, 256, 256])
{'bbox_0': {'class_id': 'animal', 'coords': [79.84808349609375, 269.634765625, 203.04376220703125, 405.0663757324219], 'threshold': 0.8010567426681519, 'pose_coords': [[56.0, 35.0], [59.0, 39.0], [59.0, 41.0], [55.0, 52.0], [58.0, 31.0], [53.0, 49.0], [55.0, 52.0], [28.0, 12.0], [25.0, 12.0], [34.0, 45.0], [23.0, 49.0], [35.0, 51.0], [14.0, 57.0], [19.0, 8.0], [10.0, 4.0], [7.0, 8.0], [11.0, 45.0], [12.0, 45.0], [23.0, 55.0], [14.0, 55.0], [9.0, 3.0], [8.0, 5.0], [9.0, 34.0]], 'pose_maxvals': [[0.2931106686592102], [0.06892652064561844], [0.5110503435134888], [0.48002466559410095], [0.029681313782930374], [0.2222478985786438], [0.4628165364265442], [0.36105087399482727], [0.425295352935791], [0.43164899945259094], [0.2479761689901352], [0.5420524477958679], [0.4040083587169647], [0.22380836308002472], [0.1389097273349762], [0.1904

TypeError: unhashable type: 'numpy.ndarray'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.7/898.7 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.0.196
    Uninstalling ultralytics-8.0.196:
      Successfully uninstalled ultralytics-8.0.196
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aja-pose 0.0.1 requires ultralytics==8.0.196, but you have ultralytics 8.3.49 which is incompatible.


In [ ]:
vid_process.get_video_metadata('/content/AnimalsGrazing/20240625_134433.mp4')

(30.01221549823788, 190, 1920, 1080)

In [ ]:

import threading
import queue

class VideoProcessor:
    def __init__(self):
        self.fit_pose = DetectionPoseInference()
        self.frame_queue = queue.Queue(maxsize=10)
        self.result_queue = queue.Queue(maxsize=10)

    def read_video(self, video_path, skip_frames=1):
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % skip_frames == 0:
                self.frame_queue.put(frame)
            frame_count += 1
        cap.release()
        self.frame_queue.put(None)  # Sentinel value to signal end of reading

    def get_video_metadata(self, video_path):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Release the video capture object
        cap.release()

        return fps, frame_count, frame_width, frame_height

    def write_video(self, output_path, fps, frame_size):
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

        while True:
            frame = self.result_queue.get()
            if frame is None:
                break
            video_writer.write(frame)

        video_writer.release()

    def process_frames(self, pose_model, detection_model, thresh, bbox_thresh):
        while True:
            frame = self.frame_queue.get()
            if frame is None:
                self.result_queue.put(None)
                break

            results = self.fit_pose.fit_pose(frame, thresh=thresh, bbox_thresh=bbox_thresh,
                                             model=pose_model, detection_model=detection_model)


            for key in results.keys():
                bbox_data = results[key]
                cords = bbox_data['coords']
                pose_coords = bbox_data['pose_coords']
                pose_maxvals = bbox_data['pose_maxvals']
                width = cords[2] - cords[0]
                height = cords[3] - cords[1]
                for i in range(len(pose_coords[0])):
                    x, y = pose_coords[0][i]
                    x, y = x * width / 64 + cords[0], y * height / 64 + cords[1]
                    if pose_maxvals[0][i] < thresh:
                        continue
                    cv2.circle(frame, (int(x), int(y)), 3, (0, 255, 0), -1)
                    cv2.rectangle(frame, (int(cords[0]), int(cords[1])), (int(cords[2]), int(cords[3])), (0, 255, 0), 2)
                    cv2.putText(frame, bbox_data['class_id'], (int(cords[0]), int(cords[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            self.result_queue.put(frame)

    def process_video(self, video_path, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', pose_model=pretrained_model,
                      thresh=0.1, bbox_thresh=0.5, output_path='video.avi', skip_frames=1):
        fps, frame_count, frame_width, frame_height = self.get_video_metadata(video_path)
        frame_size = (frame_width, frame_height)

        reader_thread = threading.Thread(target=self.read_video, args=(video_path, skip_frames))
        writer_thread = threading.Thread(target=self.write_video, args=(output_path, fps, frame_size))
        processor_thread = threading.Thread(target=self.process_frames,
                                            args=(pose_model, detection_model, thresh, bbox_thresh))

        reader_thread.start()
        processor_thread.start()
        writer_thread.start()

        reader_thread.join()
        processor_thread.join()
        writer_thread.join()

class ImageDirBatchProcessing:
    def __init__(self):
        self.det_pose = DetectionPoseInference()

    def process_images(self, image_dir, detection_model='/content/MegaDetector/MDV6b-yolov9c.pt', pose_model=pretrained_model,
                       thresh=0.1, bbox_thresh=0.5, batch_size=4, num_workers=4):
        images = os.listdir(image_dir)

        # load the data
        image_loader = torch.utils.data.DataLoader(images, batch_size=batch_size,
                                                    shuffle=False, num_workers=num_workers)

        for batch in image_loader:
            results = []
            for image in batch:
                image_path = os.path.join(image_dir, image)
                result = self.det_pose.fit_pose(image_path, thresh=thresh, bbox_thresh=bbox_thresh,
                                                model=pose_model, detection_model=detection_model)
                results.append(result)

        return results

video_folder = '/content/AnimalsGrazing'
video_files = [os.path.join(video_folder, f) for f in os.listdir(video_folder) if f.endswith('.mp4') and f != '20240625_134814.mp4']
print(video_files[0])
# inference on the first input
vid_process = VideoProcessor()
vid_process.process_video(video_path=video_files[0])


ERROR:root:Error processing [[[ 94  99  92]
  [ 83  88  81]
  [ 83  88  81]
  ...
  [ 95 125 125]
  [ 89 119 119]
  [ 84 114 114]]

 [[ 80  85  78]
  [ 69  74  67]
  [ 71  76  69]
  ...
  [ 85 115 115]
  [ 89 119 119]
  [ 91 121 121]]

 [[ 83  88  81]
  [ 73  78  71]
  [ 77  82  75]
  ...
  [ 77 104 105]
  [ 86 113 114]
  [ 92 119 120]]

 ...

 [[ 61  88  89]
  [ 62  89  90]
  [ 61  88  89]
  ...
  [199 220 219]
  [189 210 209]
  [184 205 204]]

 [[ 65  92  93]
  [ 65  92  93]
  [ 58  85  86]
  ...
  [198 219 218]
  [190 211 210]
  [184 205 204]]

 [[ 54  81  82]
  [ 56  83  84]
  [ 58  85  86]
  ...
  [185 206 205]
  [192 213 212]
  [199 220 219]]]: Can't get attribute 'RepNCSPELAN4' on <module 'ultralytics.nn.modules.block' from '/usr/local/lib/python3.10/dist-packages/ultralytics/nn/modules/block.py'>
Exception in thread Thread-12 (process_frames):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr

/content/AnimalsGrazing/20240625_134535.mp4


KeyboardInterrupt: 

In [ ]:
# prompt: save the output_json dict to a json file with indent 4 and download the file

# Assuming output_json is your dictionary
with open('output.json', 'w') as f:
  json.dump(output_json, f, indent=4)

from google.colab import files
files.download('output.json')


{'covergushter.jpg': {'bbox_0': {'class_id': 'bird',
   'coords': [75, 41, 829, 493],
   'threshold': 0.75,
   'pose_coords': [[[48.0, 11.0],
     [58.0, 7.0],
     [54.0, 14.0],
     [62.0, 15.0],
     [60.0, 15.0],
     [52.0, 21.0],
     [62.0, 15.0],
     [17.0, 27.0],
     [39.0, 31.0],
     [27.0, 6.0],
     [30.0, 38.0],
     [40.0, 41.0],
     [35.0, 49.0],
     [31.0, 27.0],
     [5.0, 31.0],
     [11.0, 41.0],
     [13.0, 24.0],
     [5.0, 60.0],
     [1.0, 30.0],
     [6.0, 60.0],
     [7.0, 35.0],
     [3.0, 34.0],
     [3.0, 36.0]]],
   'pose_maxvals': [[[0.5687083005905151],
     [0.5039724707603455],
     [0.8439555168151855],
     [0.771271288394928],
     [0.2268906533718109],
     [0.7824524641036987],
     [0.764846920967102],
     [0.20970472693443298],
     [0.3204960823059082],
     [0.1887008249759674],
     [0.3380075693130493],
     [0.2866230010986328],
     [0.6593309044837952],
     [0.16061873733997345],
     [0.4890536069869995],
     [0.4339272677898407],

This is the end

In [ ]:
from aja_pose import Model

import os
import yaml
import sys
import warnings
import os.path as osp

import torch
import torchvision.transforms as transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO
import json
import datetime

warnings.filterwarnings("ignore")
# get the current path
module_dir = os.getcwd()


 # yaml file path
yaml_file = os.path.join(module_dir, 'aja_pose', 'experiments', 'mpii', 'vhrbirdpose', 'w32_256x256_adam_lr1e-3_ak_vhr_s.yaml')
pretrained_model = os.path.join(module_dir, 'aja_pose', 'modelDir', 'all_animals_no_pretrain_106.pth')
yolo_model = os.path.join(module_dir, 'aja_pose', 'modelDir', 'yolo_det_best.pt')

def add_path(path):
    if path not in sys.path:
        sys.path.insert(0, path)

# this_dir = osp.dirname(__file__)

lib_path = osp.join(module_dir, 'aja_pose', 'lib')
add_path(lib_path)

import models

from aja_pose.lib.config import cfg
from aja_pose.lib.core.inference import *
from aja_pose.lib.utils.transforms import transform_preds
# Import the Sort Algorithm
# from aja_pose.sort import Sort
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

In [ ]:
class Args:
    def __init__(self, cfg, opts=[],modelDir='', logDir='', dataDir='', prevModelDir=''):
        self.cfg = cfg
        self.opts = opts
        self.modelDir = modelDir
        self.logDir = logDir
        self.dataDir = dataDir
        self.prevModelDir = prevModelDir


# path to data directory
class PoseInference(Model): # Inherit from the Model class
    # add path to lib directory
    def add_path(self):
        lib_path = osp.join(module_dir, 'lib')
        if lib_path not in sys.path:
            sys.path.insert(0, lib_path)

    def load_model(self, pretrained=pretrained_model):
        sys.path.append(os.path.join(module_dir, 'lib'))
        from aja_pose.lib.config import cfg
        from aja_pose.lib.config import update_config


        super().write_yaml(yaml_file, pretrained=pretrained)

        if torch.cuda.is_available():
            device = torch.device('cuda')
        else:
            device = torch.device('cpu')

        args = Args(yaml_file)

        update_config(cfg, args)

        # load the model
        model = eval('models.'+cfg.MODEL.NAME+'.get_pose_net')(
            cfg, is_train=False
        )

        # load the model weights
        model.load_state_dict(torch.load(cfg.TEST.MODEL_FILE, map_location=device), strict=False)
        model.eval()

        return model
    # load image
    def load_image(self, image_path, transform=True):
        # check if image path is a string or an image
        if isinstance(image_path, str):
            # load the image
            image = Image.open(image_path).convert('RGB')
        elif isinstance(image_path, np.ndarray):
            image = Image.fromarray(image_path)
        elif isinstance(image_path, Image.Image):
            image = image_path
        else:
            raise ValueError('Image path must be a string or an image')
         # image transformation
        image_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        if transform:
            image = image_transform(image)
            # add batch dimension
            image = image.unsqueeze(0)
        return image

    # Image inference
    def image_pose(self, image_path, model=pretrained_model, thresh=0.5):
        # load the model
        model = self.load_model(pretrained=model)
        # load the transformed image
        image_t = self.load_image(image_path)
        # load the original image
        image = self.load_image(image_path, transform=False)
        image_size = image.size
        # get the keypoints from the model
        with torch.no_grad():
            output = model(image_t) # shape (1,23,64,64) Heatmap
                # get the image centre
        def get_image_center():
            return 128,128

        # get the image scale with respect to 200px
        def get_image_scale():
            from math import sqrt
            width = 256
            height = 256
            return 200/256

        center = list(get_image_center())
        scale = get_image_scale()

        cent = np.array([[center]])
        sc = [scale]
        coords, maxvals = get_max_preds(output.clone().cpu().numpy())

        return coords, maxvals

    def batch_processing(self, images, thresh=0.5, model=pretrained_model):
        """
        Performs pose estimation on a batch of images.

        Args:
            images (torch.Tensor): A batch of images.
            thresh (float, optional): Confidence threshold for keypoints. Defaults to 0.5.
            model (torch.nn.Module, optional): Pose estimation model. Defaults to self.model.

        Returns:
            tuple: A tuple of two lists:
                - all_coords: List of keypoint coordinates for each image in the batch.
                - all_maxvals: List of keypoint confidence scores for each image in the batch.
        """

        if model is None:
            model = self.model

        with torch.no_grad():
            output = model(images)
            coords, maxvals = get_max_preds(output.clone().cpu().numpy())

        return coords, maxvals


    def draw_pose(self, image_path, thresh=0.1, model=pretrained_model):
        # load the image
        image = self.load_image(image_path, transform=False)
        # get the coords and maxvals
        coords, maxvals = self.image_pose(image_path, thresh=thresh, model=model)
        plt.imshow(image)
        plt.title('Image and predicted keypoints')
        plt.axis('off')

        # Plot the keypoints
        for i in range(len(coords[0])):  # Iterate through each keypoint
            x, y = coords[0][i]
            # x,y = x/256*image_size[0], y/256*image_size[1]
            x,y = x*image.size[0]/64, y*image.size[1]/64
            # print(x,y)
            if maxvals[0][i] < thresh:
                continue

            plt.scatter(x,y, color='red', s=10)  # Plot the keypoint on the image
        plt.show()

# object detection
class ObjectDetection:
    def load_model(self, model_path='yolov8s.pt'):
        # Add the option to load to GPU if available
        if torch.cuda.is_available():
            device = torch.device('cuda')
        else:
            device = torch.device('cpu')
        # Load the model
        model = YOLO(model_path)
        model.to(device)
        return model

    def image_detection(self, image_path, model='yolov8s.pt', stream=False):
        model = self.load_model(model)
        # load image
        if isinstance(image_path, str):
            image = Image.open(image_path).convert('RGB')
        elif isinstance(image_path, np.ndarray):
            image = Image.fromarray(image_path)
        elif isinstance(image_path, Image.Image):
            image = image_path
        # image = Image.open(image_path).convert('RGB')
        results = model.predict(image, stream=stream)
        return results

class DetectionPoseInference(ObjectDetection):
    def __init__(self):
        self.det_model = self.load_model()
        self.pose_model = PoseInference()

    def detect_objects(self, image_path, model='yolov8s.pt', stream=False):
        results = self.image_detection(image_path, model=model, stream=stream)
        return results[0]

    def _process_bbox(self, args):
        image, box, result, thresh, bbox_thresh, model = args
        bbox_data = dict()
        class_id = result.names[box.cls[0].item()]
        cords = box.xyxy[0].tolist()
        cords = [round(x) for x in cords]
        conf = round(box.conf[0].item(), 2)

        if conf < bbox_thresh:
            return None

        x1, y1, x2, y2 = cords
        cropped_image = image.crop((int(x1), int(y1), int(x2), int(y2)))
        coords, maxvals = self.pose_model.image_pose(cropped_image, thresh=thresh, model=model)
        bbox_data['class_id'] = class_id
        bbox_data['coords'] = cords
        bbox_data['threshold'] = conf
        bbox_data['pose_coords'] = coords
        bbox_data['pose_maxvals'] = maxvals

        return bbox_data

    def fit_pose(self, image_path, thresh=0.5, bbox_thresh=0.5, model='pretrained_model',
                 stream=False, detection_model='yolov8s.pt'):

        # get the bounding boxes
        result = self.detect_objects(image_path, stream=stream, model=detection_model)

        # can be an image or a video frame check if string or image
        if isinstance(image_path, str):
            image = Image.open(image_path).convert('RGB')
        elif isinstance(image_path, np.ndarray):
            image = Image.fromarray(image_path)
        elif isinstance(image_path, Image.Image):
            image = image_path

        # Prepare arguments for multiprocessing
        args = [(image, box, result, thresh, bbox_thresh, model) for box in result.boxes]

        # Use multiprocessing to process bounding boxes in parallel
        with multiprocessing.Pool() as pool:
            results = pool.map(self._process_bbox, args)

        # Collect results
        frame_data = {f'bbox_{i}': bbox_data for i, bbox_data in enumerate(results) if bbox_data is not None}

        return frame_data


    def draw_detection_pose(self, image_path, thresh=0.1, bbox_thresh=0.5, model=pretrained_model):
        # load the image
        image = Image.open(image_path).convert('RGB')
        # get the frame data for all the bounding boxes
        frame_data = self.fit_pose(image_path, thresh=thresh, bbox_thresh=bbox_thresh, model=model)
        plt.imshow(image)
        plt.title('Image and predicted keypoints')
        plt.axis('off')
        print('Bounding boxes detected:', len(frame_data))
        print('---')
        print('Bounding box data:', frame_data)

        # get the data for each bounding box in the frame and plot all the keypoints for each bounding box as well
        for key in frame_data.keys():
            print('Bounding box:', key)
            bbox_data = frame_data[key]
            class_id = bbox_data['class_id']
            cords = bbox_data['coords']
            pose_coords = bbox_data['pose_coords']
            pose_maxvals = bbox_data['pose_maxvals']
            print('Class:', class_id)
            print('Bounding box:', cords)
            print('Pose keypoints:', pose_coords)
            print('Pose maxvals:', pose_maxvals)
            print('---')
            # get the width and height of the bounding box
            width = cords[2] - cords[0]
            height = cords[3] - cords[1]
            # Plot the keypoints
            for i in range(len(pose_coords[0])):  # Iterate through each keypoint
                x, y = pose_coords[0][i]
                print(x,y)
                print('---')
                print(pose_maxvals[0][i])
                x,y = x*width/64 + cords[0], y*height/64 + cords[1]

                if pose_maxvals[0][i] < thresh:
                    continue
                plt.scatter(x,y, color='red', s=10)  # Plot the keypoint on the image
        plt.show()

# Add a class to track the object detection and pose detection (ObjectTracking)
# Track objects as they move in the frame
class ObjectTracking:
    def __init__(self):
        # initialize the object detection and pose detection classes
        self.det_pose = DetectionPoseInference()
        # self.tracker = Sort()  # Initialize the tracker

    # Read video frames or live feed
    def read_video(self, video_path): # video_path is the path to the video or 0/1 for live feed
        cap = cv2.VideoCapture(video_path)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                raise ValueError('Error reading video')
            yield frame
        cap.release()

    # Track objects in the frame
    def track_object(self, video_path, detection_model='yolov8s.pt', pose_model=pretrained_model,
                     thresh=0.1, bbox_thresh=0.5):
        # get the video frames
        video = self.read_video(video_path)
        # Define the output video path
        output_path = 'video.avi'

        # Define the video codec and create a VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        fps = 30  # Frames per second
        frame_size = (640, 480)  # Frame size (width, height)
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

        # Iterate over the frames and write them to the video file
        # for frame in video:

        for frame in video:
            # every frame goes through the detection model
            results = self.det_pose.fit_pose(frame, thresh=thresh, bbox_thresh=bbox_thresh,
                                             model=pose_model, detection_model=detection_model) # dict of bounding boxes and keypoints
            # dets - a numpy array of detections in the format [[x1,y1,x2,y2,score],[x1,y1,x2,y2,score],...]
            dets = np.empty((0,5))
            for key in results.keys():
                bbox_data = results[key]
                cords = bbox_data['coords']
                x1, y1, x2, y2 = cords
                score = bbox_data['threshold']
                dets = np.vstack((dets, [x1, y1, x2, y2, score]))
            # get the tracking results
            tracks = self.tracker.update(dets) # sending detections to the tracker func
            # iterate over each track and draw the bounding box
            for track in tracks:
                x1, y1, x2, y2, track_id = track
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                cv2.putText(frame, str(track_id), (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            video_writer.write(frame)

                # cv2.imshow('Tracking', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        # Release the video writer and close the video file
        video_writer.release()
        cv2.destroyAllWindows()

# Class to process videos
class VideoProcessor:
    def __init__(self):
        self.fit_pose = DetectionPoseInference()

    # Read video frames or live feed
    def read_video(self, video_path): # video_path is the path to the video or 0/1 for live feed
        cap = cv2.VideoCapture(video_path)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                raise ValueError('Error reading video')
            yield frame
        cap.release()

    # get video metadata
    def get_video_metadata(self, video_path):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Release the video capture object
        cap.release()

        return fps, frame_count, frame_width, frame_height

    def _process_frame(self, frame_no, frame, detection_model, pose_model, thresh, bbox_thresh):
        results = self.fit_pose(frame, thresh=thresh, bbox_thresh=bbox_thresh, model=pose_model, detection_model=detection_model)
        return frame_no, results

    # Process video frames
    def process_video(self, video_path, detection_model='yolov8s.pt', pose_model=pretrained_model,
                      thresh=0.1, bbox_thresh=0.5, output_path='video.avi'):

        # store frame data
        video_frames = dict()
        # get the video frames
        video = self.read_video(video_path)
        # Use ProcessPoolExecutor to parallelize frame processing
        with ProcessPoolExecutor() as executor:
            futures = []
            for no, frame in enumerate(video):
                futures.append(executor.submit(self._process_frame, no, frame, detection_model, pose_model, thresh, bbox_thresh))

            for future in as_completed(futures):
                frame_no, results = future.result()
                video_frames[f'frame_{frame_no}'] = results

        return video_frames

    # display frame
    def display_frame(self, video_path, frame_no=None, display=False, fps=False):
        '''
        Input:
            frame_no: int
                The frame number to display
            video_path: str
                The path to the video
        Output:
            frame: np.array
        '''
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
        success, frame = cap.read()
        # check if the frame is read
        if not success:
            raise ValueError('The frame cannot be read')
        # get the fps
        if fps:
            fps = cap.get(cv2.CAP_PROP_FPS)
            # get the number of frames
            num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            # release the video
            cap.release()
            return fps, num_frames
        if display:
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
        else:
            return frame

    # write to video/ file
    def write_data(self, video_frames, video_path=False, json_path=False, output_path='video/file_path'):
        # write the frames on top of the video
        if video_path:
            # video = self.read_video(video_path)

            # Define the video codec and create a VideoWriter object
            fourcc = cv2.VideoWriter_fourcc(*'XVID')

            # read the fps and frame size from the video
            fps, frame_count, frame_width, frame_height = self.get_video_metadata(video_path)
            frame_size = (frame_width, frame_height)  # Frame size (width, height)
            video_writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

            # write the video_frames for each frame to the video file
            for frame_no, frame_val in video_frames.items():
                # get the frame
                frame_ = frame_no.split('_')[1]
                frame = self.display_frame(video_path, frame_no=frame_)
                # Draw the bounding boxes and keypoints on the frame
                for key in frame_val.keys():
                    bbox_data = frame_val[key]
                    cords = bbox_data['coords']
                    pose_coords = bbox_data['pose_coords']
                    pose_maxvals = bbox_data['pose_maxvals']
                    # get the width and height of the bounding box
                    width = cords[2] - cords[0]
                    height = cords[3] - cords[1]
                    # Plot the keypoints
                    for i in range(len(pose_coords[0])):
                        x, y = pose_coords[0][i]
                        x,y = x*width/64 + cords[0], y*height/64 + cords[1]
                        # if pose_maxvals[0][i] < thresh:
                        #     continue
                        cv2.circle(frame, (int(x), int(y)), 3, (0, 255, 0), -1)
                        cv2.rectangle(frame, (int(cords[0]), int(cords[1])), (int(cords[2]), int(cords[3])), (0, 255, 0), 2)
                        # put text on top of the bounding box
                        cv2.putText(frame, bbox_data['class_id'], (int(cords[0]), int(cords[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                video_writer.write(frame)
                # cv2.imshow('Tracking', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            # Release the video writer and close the video file
            video_writer.release()
            cv2.destroyAllWindows()

        elif json_path:
            # write to json file
            # Save the landmarks to a JSON file
            if not os.path.exists('landmarks'):
                os.mkdir('landmarks')
            # write the landmarks dict to a JSON file with the current timestamp
            time_now = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
            with open(f'landmarks/landmarks_{time_now}.json', 'w') as f:
                json.dump(video_frames, f, indent=4)


inf = PoseInference()
obj_det = ObjectDetection()
pose_det = DetectionPoseInference()
obj_track = ObjectTracking()
vid_process = VideoProcessor()

In [ ]:
import os

# data directory
dir_data_path = os.path.join(module_dir, 'aja_pose', 'dataDir')
# image path
img_path = os.path.join(dir_data_path, 'licensed-image.jfif')
# video path
vid_path = os.path.join(dir_data_path, 'llama2.mp4')

In [ ]:
vid_process.write_data(video_frames = vid_process.process_video(vid_path, detection_model='yolov8l.pt', pose_model=pretrained_model,
                            thresh=0.1, bbox_thresh=0.5, output_path='sheep_grazing_det.avi'), json_path='k.json')

Process ForkProcess-522:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
  File "/usr/local/lib/python3.10/dist-packages/torch/multiprocessing/reductions.py", line 147, in rebuild_cuda_tensor
    torch.cuda._lazy_init()
  File "/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py", line 288, in _lazy_init
    raise RuntimeError(
RuntimeError: Cannot re-initialize CUDA in forked subprocess. To use CUDA with multiprocessing, you must use the 'spawn' start method
Process ForkProcess-523:
Traceback (most recent call last):
  File "/usr/lib/py

In [ ]:
video_frames = vid_process.process_video(vid_path, detection_model='yolov8l.pt', pose_model=pretrained_model,
                            thresh=0.1, bbox_thresh=0.5, output_path='llama2.avi')

100%|██████████| 83.7M/83.7M [00:00<00:00, 272MB/s]



0: 640x384 1 horse, 1 sheep, 127.8ms
Speed: 15.6ms preprocess, 127.8ms inference, 493.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 horse, 1 sheep, 40.4ms
Speed: 1.9ms preprocess, 40.4ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 horse, 1 sheep, 40.5ms
Speed: 1.8ms preprocess, 40.5ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 horse, 2 sheeps, 41.0ms
Speed: 2.6ms preprocess, 41.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 sheeps, 40.8ms
Speed: 1.7ms preprocess, 40.8ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 sheeps, 41.0ms
Speed: 2.5ms preprocess, 41.0ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 horse, 2 sheeps, 47.0ms
Speed: 2.5ms preprocess, 47.0ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 horse, 2 sheeps, 40.7ms
Speed: 1.8ms preprocess, 40.7ms i

ValueError: Error reading video

In [ ]:
!ls

AJA-pose  dataset  dataset.zip	sample_data  yolov8s.pt


In [ ]:
!mkdir /content/AJA-pose/code/aja_pose/modelDir

Remember to Restart session before running the next cell to ensure the package is installed

In [ ]:
from aja_pose import Model

# path to the images directory and annotation in mpii json format
images_directory = 'dataset'
mpii_json = 'custom_test.json' # Test done on custom_test
model_file = 'all_animals_no_pretrain_106.pth' # Current best model

# Initialize the class
model = Model()

In [ ]:
# Test the model on custom data
model.test(images_directory, mpii_json, model_file)

Performing Tests on the Animal Kingdom Dataset

In [ ]:
# Test the model on Protocol 1
model.test(images_directory, protocol='P1', model=model_file)
# Test the model on Protocol 2
model.test(images_directory, protocol='P2', model=model_file)
# Test the model on birds class Protocol 3
model.test(images_directory, protocol='P3', model=model_file, animal_class='bird')
# Test the model on reptiles class Protocol 3
model.test(images_directory, protocol='P3', model=model_file, animal_class='reptile')
# Test the model on mammals class Protocol 3
model.test(images_directory, protocol='P3', model=model_file, animal_class='mammal')
# Test the model on fish class Protocol 3
model.test(images_directory, protocol='P3', model=model_file, animal_class='fish')
# Test the model on amphibian class Protocol 3
model.test(images_directory, protocol='P3', model=model_file, animal_class='amphibian')

Training your own model(VHR) or pretraining on our weights

In [ ]:
# train a VHR model
images_directory = '' # Ptath to images directory
train_json = 'train.json' # labels for the train set
valid_json = 'test.json' # Labels for the validation set
model_file = '' # A pytorch model file to pretrain on.
model.train(images_directory, train_json, valid_json, pretrained=model_file)